In [1]:
# Importando as bibliotecas necessárias

import pyspark.sql.functions as F
from pyspark.sql.window import Window

StatementMeta(, 7c448164-6686-4a43-a93a-782512ad0ad9, 3, Finished, Available, Finished, False)

In [4]:
# Leitura da tabela cotacoes

df_silver = spark.read.table("cotacoes")

# Mínimas datas por moeda (CTE: base_dados)

df_base_dados = df_silver.groupBy("Moeda").agg(
    F.min("Data").alias("min_data")
)

# Gera todas as datas do período (CTE: datas_periodo)
# Usamos o sequence para criar um array de datas e o explode para transformar o array em linhas

df_datas_periodo = df_base_dados.withColumn(
    "Data_Array", 
    F.sequence(F.col("min_data"), F.current_date(), F.expr("interval 1 day"))
).withColumn(
    "Data", 
    F.explode(F.col("Data_Array"))
).select("Moeda", "Data")

# Mescla as cotações atuais (CTE: dados_completos)
# O uso de uma lista no parâmetro 'on' evita colunas duplicadas no resultado final do PySpark

df_dados_completos = df_datas_periodo.join(
    df_silver,
    on=["Moeda", "Data"],
    how="left"
)

# Preenche as lacunas com a última cotação (CTE: ft_cotacoes)
# Definindo a partição e a janela de dados

window_spec = Window.partitionBy("Moeda") \
                    .orderBy("Data") \
                    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Aplicando o last_value ignorando os nulos (ignorenulls=True equivale ao "true" no SQL)

df_ft_cotacoes = df_dados_completos.withColumn(
    "Cotacao",
    F.last("Cotacao", ignorenulls=True).over(window_spec)
)

# Ordenação final e Escrita na camada Gold
# Mantive o seu ORDER BY, mas avalie se ele é realmente estritamente necessário para o seu caso de uso

df_final = df_ft_cotacoes.orderBy("Data", "Moeda")

# Salvando como Delta Table no Lakehouse
# O mode "overwrite" com "overwriteSchema" replica o comportamento do "CREATE OR REPLACE"

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ft_cotacoes")

display(df_final)

StatementMeta(, 7c448164-6686-4a43-a93a-782512ad0ad9, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e41a2f7d-dd92-4612-9db7-6fc5d46bf817)

In [1]:
%%sql

SELECT * FROM ft_cotacoes

StatementMeta(, 3ed0834f-6d01-4d94-b2b0-0b7613aa5009, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 3 fields>